# Análise Exploratória de Dados

### Importação de bibliotecas e do conjunto de dados

In [47]:
import pandas as pd
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [48]:
df_pedidos = pd.read_csv("../data/demanda_pedidos_final.csv")
df_pedidos.head()

,id_produto,quantidade,data_pedido
0,1206997678774,2.00,2025-12-31
1,2483458744482,2.00,2025-12-31
2,3115403067986,2.00,2025-12-31
3,4616333531350,2.00,2025-12-31
4,2124623355660,5.00,2025-12-31


In [49]:
df_produtos = pd.read_csv("../data/demanda_produtos_final.csv")
df_produtos.head()

,id_produto,curva,categoria,fora_de_estoque,indice_de_preco,data_coleta
0,6693862302648,B,Categoria 35,0,0.95,2026-03-12
1,6693862302648,B,Categoria 35,0,0.95,2026-03-13
2,6693862302648,B,Categoria 35,0,1.18,2026-03-14
3,6693862302648,B,Categoria 35,0,1.18,2026-03-15
4,6693862302648,B,Categoria 35,0,0.95,2026-03-16


Com o objetivo de adequar o conjunto de dados à especificação original mostrada pela equipe da Praso, nós decidimos modificar a coluna `fora_de_estoque`, que indica a falta do produto no estoque naquela data (1) ou não (0), para uma coluna exatamente oposta à esta, chamada `estoque`, que indica se o produto tinha estoque na data (1) ou não (0).

In [50]:
# Ajustando tabela de produtos pra ficar igual à especificação
df_produtos["fora_de_estoque"] = df_produtos["fora_de_estoque"].replace({0: 1, 1: 0})
df_produtos.rename(columns={"fora_de_estoque": "estoque"}, inplace=True)

### Análise Exploratória Estrutural

#### Dataset de pedidos

In [51]:
print("====== INFORMAÇÕES GERAIS ======")
print(f"Formato: {df_pedidos.shape[0]} linhas e {df_pedidos.shape[1]} colunas")
print(f"Uso de memória: {df_pedidos.memory_usage(deep=True).sum() / 1024**2:.2f} Mb")

====== INFORMAÇÕES GERAIS ======
Formato: 45807 linhas e 3 colunas
Uso de memória: 3.28 Mb


In [52]:
print("====== INFORMAÇÕES DETALHADAS ======")
print(df_pedidos.info())

====== INFORMAÇÕES DETALHADAS ======
<class 'pandas.DataFrame'>
RangeIndex: 45807 entries, 0 to 45806
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id_produto   45807 non-null  int64  
 1   quantidade   45807 non-null  float64
 2   data_pedido  45807 non-null  str    
dtypes: float64(1), int64(1), str(1)
memory usage: 1.0 MB
None


In [53]:
print("====== ESTATÍSTICAS DESCRITIVAS ======")
print(df_pedidos.describe())

====== ESTATÍSTICAS DESCRITIVAS ======
            id_produto  quantidade
count         45807.00    45807.00
mean  4961739290666.05        9.92
std   2589518184649.85       23.67
min   1010954545613.00        1.00
25%   2826923850851.00        1.00
50%   4585589272177.00        3.00
75%   6997563345298.00       12.00
max   9986625254736.00     2400.00


Apesar de não ser um atributo numérico propriamente dito, `id_produto` entra nas estatísticas descritivas do método `describe()` por ser do tipo `int64`.

In [54]:
print("====== VALORES ÚNICOS POR COLUNA ======")
for col in df_pedidos.columns:
    unique = df_pedidos[col].nunique()
    print(f"{col:20}{unique}")

====== VALORES ÚNICOS POR COLUNA ======
id_produto          340
quantidade          125
data_pedido         92


### Dataset de produtos

In [55]:
print("====== INFORMAÇÕES GERAIS ======")
print(f"Formato: {df_produtos.shape[0]} linhas e {df_produtos.shape[1]} colunas")
print(f"Uso de memória: {df_produtos.memory_usage(deep=True).sum() / 1024**2:.2f} Mb")

====== INFORMAÇÕES GERAIS ======
Formato: 6453 linhas e 6 colunas
Uso de memória: 1.19 Mb


In [56]:
print("====== INFORMAÇÕES DETALHADAS ======")
print(df_produtos.info())

====== INFORMAÇÕES DETALHADAS ======
<class 'pandas.DataFrame'>
RangeIndex: 6453 entries, 0 to 6452
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_produto       6453 non-null   int64  
 1   curva            6453 non-null   str    
 2   categoria        6453 non-null   str    
 3   estoque          6453 non-null   int64  
 4   indice_de_preco  5628 non-null   float64
 5   data_coleta      6453 non-null   str    
dtypes: float64(1), int64(2), str(3)
memory usage: 302.6 KB
None


In [57]:
print("====== ESTATÍSTICAS DESCRITIVAS ======")
print(df_produtos.describe())

====== ESTATÍSTICAS DESCRITIVAS ======
            id_produto  estoque  indice_de_preco
count          6453.00  6453.00          5628.00
mean  4913289714151.82     0.85             1.11
std   2604027767656.01     0.36             0.32
min   1010954545613.00     0.00             0.21
25%   2672346263770.00     1.00             0.98
50%   4585589272177.00     1.00             1.07
75%   6501363631104.00     1.00             1.15
max   9952616835809.00     1.00             3.52


Da mesma forma em que no outro dataset, `id_produto` também entrou nas estatísticas descritivas.

In [58]:
print("====== VALORES ÚNICOS POR COLUNA ======")
for col in df_produtos.columns:
    unique = df_produtos[col].nunique()
    print(f"{col:20}{unique}")

====== VALORES ÚNICOS POR COLUNA ======
id_produto          103
curva               2
categoria           37
estoque             2
indice_de_preco     925
data_coleta         91


### Análise de valores faltantes

#### Dataset de pedidos

In [59]:
print("====== VALORES FALTANTES POR COLUNA ======")
missing_pedidos = pd.DataFrame({
    'coluna': df_pedidos.columns,
    'missing_count': df_pedidos.isnull().sum(),
    'missing_percent': (df_pedidos.isnull().sum()/len(df_pedidos)) * 100
})

print(missing_pedidos)

====== VALORES FALTANTES POR COLUNA ======
                  coluna  missing_count  missing_percent
id_produto    id_produto              0             0.00
quantidade    quantidade              0             0.00
data_pedido  data_pedido              0             0.00


Como não existem valores faltantes, podemos seguir para a próxima análise.

#### Dataset de produtos

In [60]:
print("====== VALORES FALTANTES POR COLUNA ======")
missing_produtos = pd.DataFrame({
    'coluna': df_produtos.columns,
    'missing_count': df_produtos.isnull().sum(),
    'missing_percent': (df_produtos.isnull().sum()/len(df_produtos)) * 100
})

print(missing_produtos)

====== VALORES FALTANTES POR COLUNA ======
                          coluna  missing_count  missing_percent
id_produto            id_produto              0             0.00
curva                      curva              0             0.00
categoria              categoria              0             0.00
estoque                  estoque              0             0.00
indice_de_preco  indice_de_preco            825            12.78
data_coleta          data_coleta              0             0.00


De acordo com a equipe da Praso, o dado de índice de preço pode ser nulo caso não tivessemos uma referência de preço do mercado na data. Portanto, podemos posteriormente realizar o procedimento de _______

Talvez, isso por si só já indique um mecanismo de Missing Completly at Random (MCAR), o que permite a técnica xxx.

In [61]:
print("====== ANÁLISE DO MECANISMO DE MISSING ======")
# Talvez nem seja necessário

====== ANÁLISE DO MECANISMO DE MISSING ======
